<a href="https://colab.research.google.com/github/andresanchez256/Portafolio/blob/main/Proyecto_Clasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Análisis de Riesgo Crediticio con Machine Learning

## 📋 Contexto del Problema

El riesgo crediticio es una de las áreas más críticas en la banca y las finanzas. Las instituciones financieras necesitan evaluar la probabilidad de que un solicitante de crédito no pague su préstamo (default) para tomar decisiones informadas.

### 🎯 Objetivo del Proyecto
Desarrollar un modelo de machine learning que prediga la **probabilidad de impago** de un cliente basado en sus características financieras y demográficas, utilizando el **German Credit Dataset**.

### 💰 Impacto en el Negocio
- **Reducción de pérdidas**: Identificar clientes de alto riesgo antes de otorgar créditos
- **Optimización de tasas**: Ajustar tasas de interés según el nivel de riesgo
- **Mejor toma de decisiones**: Complementar el juicio humano con predicciones basadas en datos

### 📊 Métricas de Éxito
- **AUC-ROC**: Capacidad del modelo para distinguir entre buenos y malos pagadores
- **Costo total**: Minimizar el costo de errores (falsos positivos y falsos negativos)
- **Interpretabilidad**: Entender qué variables influyen más en el riesgo

### 📚 Dataset Utilizado
- **Fuente**: UCI Machine Learning Repository - German Credit Dataset
- **Registros**: 1000 solicitudes de crédito
- **Variables**: 24 características (demográficas, financieras, historial crediticio)
- **Clases**: 700 "Buenos pagadores" (70%) y 300 "Malos pagadores" (30%)

In [1]:
# ============================================
# CONFIGURACIÓN INICIAL DEL NOTEBOOK
# ============================================

# Instalar librerías necesarias (solo si no están instaladas)
!pip install -q pandas numpy scikit-learn matplotlib seaborn

# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.metrics import precision_recall_curve, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("✅ Librerías importadas correctamente")
print(f"📊 Pandas versión: {pd.__version__}")
print(f"🔢 NumPy versión: {np.__version__}")

✅ Librerías importadas correctamente
📊 Pandas versión: 2.2.2
🔢 NumPy versión: 2.0.2


## 📊 Exploración del Dataset

### Descripción de Variables

El German Credit Dataset contiene 24 variables que describen al solicitante del crédito:

#### Variables Demográficas
- **age**: Edad del solicitante en años
- **personal_status_sex**: Estado civil y género
- **foreign_worker**: Indicador de trabajador extranjero

#### Variables Financieras
- **amount**: Monto solicitado del préstamo (DM)
- **duration_months**: Duración del préstamo en meses
- **installment_rate**: Porcentaje del ingreso destinado a la cuota
- **status_account**: Estado de la cuenta corriente

#### Historial Crediticio
- **credit_history**: Historial de créditos anteriores
- **existing_credits_count**: Número de créditos existentes
- **savings_account**: Estado de la cuenta de ahorros

#### Variables Laborales
- **employment_duration**: Antigüedad en el empleo actual
- **job**: Tipo de ocupación

### Variable Objetivo
- **target**: 0 = Buen pagador, 1 = Mal pagador (impago)

### 🔍 Estructura del Problema
- **Tipo**: Clasificación binaria
- **Desbalanceo**: 70% buenos, 30% malos
- **Contexto**: Matriz de costos (FN = 5, FP = 1)

In [ ]:
# ============================================
# CARGA DE DATOS
# ============================================

# URL del dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data-numeric'

# Definir nombres de variables (23 features)
feature_names = [
    'status_account',      # Estado de cuenta corriente
    'duration_months',     # Duración del préstamo (meses)
    'credit_history',      # Historial crediticio
    'purpose',             # Propósito del préstamo
    'amount',              # Monto solicitado (DM)
    'savings_account',     # Cuenta de ahorros
    'employment_duration', # Antigüedad laboral
    'installment_rate',    # Tasa de cuota (% ingreso)
    'personal_status_sex', # Estado civil y género
    'other_debtors',       # Otros deudores
    'residence_since',     # Años en residencia
    'property',            # Propiedad
    'age',                 # Edad (años)
    'other_installment_plans', # Otros planes de cuota
    'housing',             # Situación de vivienda
    'existing_credits_count', # Número de créditos existentes
    'job',                 # Tipo de trabajo
    'people_liable',       # Personas a cargo
    'telephone',           # Teléfono
    'foreign_worker',      # Trabajador extranjero
    'feature_21',          # Variable adicional
    'feature_22',          # Variable adicional
    'feature_23'           # Variable adicional
]

# Cargar los datos
df = pd.read_csv(url, sep='\s+', header=None)

# Asignar nombres de columnas (23 features + 1 clase)
df.columns = feature_names + ['class']

# Crear variable target (0 = buen pagador, 1 = mal pagador)
df['target'] = df['class'].apply(lambda x: 0 if x == 1 else 1)

# Eliminar columna 'class' original
df = df.drop('class', axis=1)

# Mostrar información básica
print(f"✅ Datos cargados exitosamente")
print(f"📊 Dimensiones: {df.shape[0]} registros, {df.shape[1]} columnas")
print(f"\n📋 Distribución de clases:")
print(f"  - Buenos pagadores (0): {sum(df['target']==0)} ({sum(df['target']==0)/len(df)*100:.1f}%)")
print(f"  - Malos pagadores (1): {sum(df['target']==1)} ({sum(df['target']==1)/len(df)*100:.1f}%)")